In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_primary = os.getenv("AZURE_OPENAI_MODEL")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure client ready")
print("Primary model:", model_primary)

Azure client ready
Primary model: gpt-4.1-mini


In [3]:
policy_question = "Explain when prior authorization is required for MRI procedures."

prompt_versions = {
    "Basic": policy_question,

    "Role + Context": """
You are a healthcare payer policy assistant.
Explain when prior authorization is required for MRI procedures
for a healthcare technology professional.
""",

    "Controlled": """
You are a healthcare payer policy assistant.

Task:
Explain when prior authorization is required for MRI procedures.

Requirements:
- Use payer terminology.
- Keep the answer concise.
- Maximum 3 bullet points.
- Do not invent policy details that are not provided.
"""
}

responses = {}

for name, prompt in prompt_versions.items():

    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    responses[name] = response

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(response.choices[0].message.content)


BASIC
------------------------------------------------------------
Prior authorization for MRI procedures is typically required by insurance providers to ensure that the imaging study is medically necessary before it is performed. The requirements can vary depending on the specific insurance plan, but generally, prior authorization is required in the following situations:

1. **Non-emergency MRI Scans:**  
   Most health insurance plans require prior authorization for elective or non-emergency MRI scans. This means if the MRI is scheduled in advance and not for an urgent or life-threatening condition, the provider usually must obtain approval from the insurer.

2. **MRI Covered Under Specific Medical Necessity Criteria:**  
   Insurance carriers often have guidelines that define when an MRI is considered medically necessary. Prior authorization ensures that the requested MRI meets these criteria, such as documentation of symptoms, prior conservative treatments, or relevant clinical fi

In [4]:
policy_question = "Explain when prior authorization is required for MRI procedures."

prompt_versions = {
    "Basic": policy_question,

    "Role + Context": """
You are a healthcare payer policy assistant.
Explain when prior authorization is required for MRI procedures
for a healthcare technology professional.
""",

    "Controlled": """
You are a healthcare payer policy assistant.

Task:
Explain when prior authorization is required for MRI procedures.

Requirements:
- Use payer terminology.
- Keep the answer concise.
- Maximum 3 bullet points.
- Do not invent policy details that are not provided.
"""
}

responses = {}

for name, prompt in prompt_versions.items():

    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    responses[name] = response

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(response.choices[0].message.content)


BASIC
------------------------------------------------------------
Prior authorization for MRI (Magnetic Resonance Imaging) procedures is typically required by insurance companies and healthcare payers to ensure that the imaging is medically necessary and appropriate before the service is performed. The exact criteria and timing can vary depending on the insurance plan, provider policies, and regulatory guidelines, but generally, prior authorization is required in the following situations:

1. **Insurance Plan Requirements:**  
   Most commercial insurance plans, Medicare Advantage plans, and Medicaid managed care plans require prior authorization for MRI scans. This is to control costs and prevent unnecessary imaging.

2. **Non-Emergent MRIs:**  
   Prior authorization is usually required for scheduled, non-emergency MRI exams. Emergency MRIs performed in acute settings (e.g., hospital emergency rooms) typically do not require prior authorization.

3. **Specific Body Regions or Condi

In [5]:
import json

few_shot_prompt = """
You are a healthcare payer policy assistant.

Return the answer as JSON with these fields:
- policy_topic
- authorization_required
- reason
- confidence

Examples:

Question: Does physical therapy require authorization after 10 visits?
Answer:
{
  "policy_topic": "Physical Therapy",
  "authorization_required": true,
  "reason": "Authorization is required after the initial 10 visits.",
  "confidence": "high"
}

Question: Is prior authorization required for emergency imaging?
Answer:
{
  "policy_topic": "Emergency Imaging",
  "authorization_required": false,
  "reason": "The provided policy states emergency imaging is exempt.",
  "confidence": "high"
}

Now answer:

Question: Does an MRI require prior authorization?

Policy context:
MRI procedures require prior authorization.
"""

response_structured = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": few_shot_prompt
        }
    ]
)

print(response_structured.choices[0].message.content)

{
  "policy_topic": "MRI",
  "authorization_required": true,
  "reason": "MRI procedures require prior authorization according to the policy.",
  "confidence": "high"
}


In [6]:
# Step 1: Extract a structured policy decision

policy_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
"""

user_question = "The provider wants to schedule a non-emergency MRI. What should happen?"

extract_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": f"""
Using only the policy below, extract the decision.

Policy:
{policy_context}

Question:
{user_question}

Return only:
Authorization Required: Yes/No
Reason:
"""
        }
    ]
)

policy_decision = extract_response.choices[0].message.content

print("STEP 1 - POLICY DECISION")
print(policy_decision)

STEP 1 - POLICY DECISION
Authorization Required: Yes  
Reason: MRI procedures require prior authorization, and this is a non-emergency case.


In [7]:
action_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare operations assistant.
Use the policy decision provided to recommend the next operational action.
Do not change or reinterpret the policy decision.
Keep the response to one concise sentence.
"""
        },
        {
            "role": "user",
            "content": f"""
Policy decision:
{policy_decision}

What should the operations team do next?
"""
        }
    ]
)

print("STEP 2 - OPERATIONAL ACTION")
print(action_response.choices[0].message.content)

STEP 2 - OPERATIONAL ACTION
The operations team should initiate the prior authorization process for the MRI procedure before scheduling.


In [8]:

full_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
CT scans require prior authorization for outpatient procedures.
Physical therapy requires authorization after 10 visits.
Specialist consultations do not require prior authorization.
Dental coverage is excluded from this plan.
Vision coverage is available once every 24 months.
Member address changes must be updated within 30 days.
"""

reduced_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
"""

question = "Does a non-emergency MRI require prior authorization?"

In [9]:
full_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{full_context}

Question:
{question}
"""
        }
    ]
)

reduced_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{reduced_context}

Question:
{question}
"""
        }
    ]
)

print("FULL CONTEXT")
print(full_response.choices[0].message.content)
print("Prompt tokens:", full_response.usage.prompt_tokens)

print("\nREDUCED CONTEXT")
print(reduced_response.choices[0].message.content)
print("Prompt tokens:", reduced_response.usage.prompt_tokens)

FULL CONTEXT
Yes, a non-emergency MRI requires prior authorization.
Prompt tokens: 104

REDUCED CONTEXT
Yes, a non-emergency MRI requires prior authorization.
Prompt tokens: 48


In [10]:
def healthcare_policy_assistant(question, policy_context):
    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "system",
                "content": """
You are a healthcare payer policy assistant.

Rules:
1. Use only the provided policy context.
2. Do not invent missing policy details.
3. If the evidence is insufficient, say "Insufficient information".
4. Keep the response concise and operational.
5. Return the answer in this structure:

Decision:
Reason:
Next Action:
"""
            },
            {
                "role": "user",
                "content": f"""
Policy Context:
{policy_context}

Question:
{question}
"""
            }
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens
    }

In [11]:
policy_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
Physical therapy requires authorization after 10 visits.
"""

question = "A provider wants to schedule a non-emergency MRI. What should happen?"

result = healthcare_policy_assistant(
    question=question,
    policy_context=policy_context
)

print(result["answer"])

print("\nTOKEN USAGE")
print("Prompt tokens     :", result["prompt_tokens"])
print("Completion tokens :", result["completion_tokens"])
print("Total tokens      :", result["total_tokens"])

Decision:
The non-emergency MRI requires prior authorization.

Reason:
MRI procedures require prior authorization unless it is an emergency imaging case, which this is not.

Next Action:
The provider should submit a prior authorization request before scheduling the MRI.

TOKEN USAGE
Prompt tokens     : 123
Completion tokens : 48
Total tokens      : 171
